<table align="center">
 <td align="center"><a target="_blank" href="https://colab.research.google.com/github/ezponda/intro_deep_learning/blob/main/class/CNN/Image_Segmentation_with_HuggingFace.ipynb">
        <img src="https://colab.research.google.com/img/colab_favicon_256px.png"  width="50" height="50" style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/Image_Segmentation_with_HuggingFace.ipynb">
        <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"  width="50" height="50" style="padding-bottom:5px;" />View Source on GitHub</a></td>
</table>

# Image Segmentation with Hugging Face

## Table of Contents

1. [Introduction to Hugging Face](#1.-Introduction-to-Hugging-Face)
2. [Image Segmentation Concepts](#2.-Image-Segmentation-Concepts)
3. [The ISIC Skin Lesion Dataset](#3.-The-ISIC-Skin-Lesion-Dataset)
4. [Preparing the Data](#4.-Preparing-the-Data)
5. [Fine-tuning SegFormer](#5.-Fine-tuning-SegFormer)
6. [Evaluation and Visualization](#6.-Evaluation-and-Visualization)
7. [What's Next](#7.-What's-Next)

## 1. Introduction to Hugging Face

[Hugging Face](https://huggingface.co/) is the most widely used platform for sharing and using machine learning models. Think of it as a GitHub for AI, with three main components:

- **Models**: over 500,000 pre-trained models for any task (text, vision, audio, multimodal). Each model has a card with documentation, metrics, and usage examples.
- **Datasets**: over 100,000 curated datasets that you can load with a single line of code using the `datasets` library.
- **Spaces**: live interactive demos where you can try models directly in your browser.

The key libraries in the Hugging Face ecosystem are:

| Library | Purpose |
|---------|---------|
| `transformers` | Load and use pre-trained models |
| `datasets` | Load and process datasets |
| `evaluate` | Compute evaluation metrics |
| `accelerate` | Simplify training on GPUs |

In this notebook, we'll use all four to **fine-tune a segmentation model** on medical images.

### The `pipeline()` API

The simplest way to use a pre-trained model is the `pipeline()` function. It handles model loading, preprocessing, inference, and postprocessing in a single call.

Let's try it with an image segmentation model:

In [ ]:
#!pip install transformers datasets evaluate accelerate torch

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import requests
import os
import glob

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
from transformers import pipeline

# One line to load a segmentation model
segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512", device=device)

# Load an image
IMG_BASE = "https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images"
response = requests.get(f"{IMG_BASE}/beach.jpg")
image = Image.open(BytesIO(response.content)).convert("RGB")

# Run segmentation
results = segmenter(image)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Overlay segmentation masks
axes[1].imshow(image)
colors = plt.cm.tab20(np.linspace(0, 1, len(results)))
for result, color in zip(results, colors):
    mask = np.array(result["mask"])
    overlay = np.zeros((*mask.shape, 4))
    overlay[mask > 0] = [*color[:3], 0.5]
    axes[1].imshow(overlay)
    ys, xs = np.where(mask > 0)
    if len(ys) > 0:
        axes[1].text(xs.mean(), ys.mean(), result["label"], fontsize=8,
                     ha="center", va="center", color="white", weight="bold",
                     bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.7))
axes[1].set_title("Segmentation (SegFormer, ADE20K)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"Detected {len(results)} segments")
for r in results[:8]:
    print(f"  {r['label']}: {r['score']:.0%}" if r['score'] else f"  {r['label']}")

This model was trained on [ADE20K](https://groups.csail.mit.edu/vision/datasets/ADE20K/) (150 indoor/outdoor categories). It works well for general scenes, but for specialized domains like **medical imaging**, we need to fine-tune it on domain-specific data.

That's exactly what we'll do in this notebook: fine-tune SegFormer to segment **skin lesions** in dermoscopic images.

## Question 1: Explore the Hugging Face Hub

Go to [huggingface.co/models](https://huggingface.co/models) and filter by task: **Image Segmentation**.

1. How many segmentation models are available?
2. Try loading a different segmentation model with `pipeline()` and run it on the beach image
3. Do you get different results? Which model produces better segmentation?

In [ ]:
# Try a different model from the Hub
different_model = ...
segmenter2 = pipeline("image-segmentation", model=different_model, device=device)
results2 = segmenter2(image)

for r in results2[:8]:
    print(f"  {r['label']}: {r['score']:.0%}" if r['score'] else f"  {r['label']}")

## 2. Image Segmentation Concepts

Image segmentation assigns a label to **every pixel** in an image. There are three main types:

| Type | Description | Example |
|------|-------------|---------|
| **Semantic** | Each pixel gets a class label. All objects of the same class share the same label. | All cars = blue, all trees = green |
| **Instance** | Like semantic, but distinguishes between different objects of the same class. | Car 1 = blue, Car 2 = red |
| **Panoptic** | Combines both: semantic labels for "stuff" (sky, road) and instance labels for "things" (cars, people). | Full scene understanding |

In this notebook we focus on **binary semantic segmentation**: each pixel is either **background** (0) or **lesion** (1).

### Evaluation Metrics

The two most common metrics for segmentation are:

**IoU (Intersection over Union)**, also called Jaccard Index:

$$\text{IoU} = \frac{|\text{Prediction} \cap \text{Ground Truth}|}{|\text{Prediction} \cup \text{Ground Truth}|}$$

**Dice Coefficient** (F1 score for segmentation):

$$\text{Dice} = \frac{2 \times |\text{Prediction} \cap \text{Ground Truth}|}{|\text{Prediction}| + |\text{Ground Truth}|}$$

Both range from 0 (no overlap) to 1 (perfect match). Dice is always >= IoU for the same prediction.

In [ ]:
def compute_iou(pred, target):
    intersection = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    return intersection / union if union > 0 else 0.0

def compute_dice(pred, target):
    intersection = np.logical_and(pred, target).sum()
    total = pred.sum() + target.sum()
    return 2 * intersection / total if total > 0 else 0.0

# Example: two overlapping circles
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Create sample masks
y, x = np.ogrid[-50:50, -50:50]
ground_truth = (x**2 + y**2 < 40**2).astype(np.uint8)

for i, (shift, title) in enumerate([(5, "Good"), (20, "Medium"), (40, "Poor")]):
    pred = ((x - shift)**2 + (y - shift)**2 < 40**2).astype(np.uint8)
    iou = compute_iou(pred, ground_truth)
    dice = compute_dice(pred, ground_truth)

    axes[i].imshow(ground_truth + pred, cmap="Blues", vmin=0, vmax=2)
    axes[i].set_title(f"{title}\nIoU: {iou:.2f} | Dice: {dice:.2f}")
    axes[i].axis("off")

plt.suptitle("IoU and Dice for different overlap levels", fontsize=13)
plt.tight_layout()
plt.show()

## 3. The ISIC Skin Lesion Dataset

The [ISIC (International Skin Imaging Collaboration)](https://www.isic-archive.com/) provides the largest public collection of dermoscopic images for skin cancer research.

We use the **ISIC 2017 Challenge** dataset (Task 1 - Lesion Segmentation):
- **2,000 training images** with expert-annotated binary masks
- **150 validation images** with masks
- Each mask marks the lesion boundary (white = lesion, black = background)

The images are dermoscopic photographs of skin lesions taken with specialized dermatoscopes.

In [ ]:
import urllib.request
import zipfile

def download_and_extract(url, expected_dir):
    if os.path.exists(expected_dir):
        print(f"  {expected_dir} already exists, skipping")
        return
    zip_name = url.split("/")[-1]
    print(f"  Downloading {zip_name}...")
    urllib.request.urlretrieve(url, zip_name)
    print(f"  Extracting...")
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(".")
    os.remove(zip_name)
    print(f"  Done!")

ISIC_BASE = "https://isic-archive.s3.amazonaws.com/challenges/2017"

print("Downloading ISIC 2017 dataset...\n")
download_and_extract(f"{ISIC_BASE}/ISIC-2017_Training_Data.zip", "ISIC-2017_Training_Data")
download_and_extract(f"{ISIC_BASE}/ISIC-2017_Training_Part1_GroundTruth.zip", "ISIC-2017_Training_Part1_GroundTruth")
download_and_extract(f"{ISIC_BASE}/ISIC-2017_Validation_Data.zip", "ISIC-2017_Validation_Data")
download_and_extract(f"{ISIC_BASE}/ISIC-2017_Validation_Part1_GroundTruth.zip", "ISIC-2017_Validation_Part1_GroundTruth")
print("\nAll files downloaded!")

In [ ]:
def get_paired_paths(image_dir, mask_dir):
    """Find all (image, mask) pairs from ISIC directories."""
    image_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    # Filter out superpixel images
    image_paths = [p for p in image_paths if "superpixel" not in p.lower()]
    pairs = []
    for img_path in image_paths:
        name = os.path.basename(img_path).replace(".jpg", "")
        mask_path = os.path.join(mask_dir, f"{name}_segmentation.png")
        if os.path.exists(mask_path):
            pairs.append((img_path, mask_path))
    return pairs

train_pairs = get_paired_paths("ISIC-2017_Training_Data", "ISIC-2017_Training_Part1_GroundTruth")
val_pairs = get_paired_paths("ISIC-2017_Validation_Data", "ISIC-2017_Validation_Part1_GroundTruth")

print(f"Training pairs: {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")

In [ ]:
# Visualize some samples with overlay
def show_samples(pairs, n=4):
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    for i in range(n):
        img = Image.open(pairs[i][0]).convert("RGB")
        mask = np.array(Image.open(pairs[i][1]).convert("L"))

        # Image
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Dermoscopic Image")

        # Mask
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title("Segmentation Mask")

        # Overlay
        axes[i, 2].imshow(img)
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask > 128] = [1, 0, 0, 0.4]  # Red overlay on lesion
        axes[i, 2].imshow(overlay)
        lesion_pct = (mask > 128).sum() / mask.size * 100
        axes[i, 2].set_title(f"Overlay (lesion: {lesion_pct:.1f}%)")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples(train_pairs, n=4)

## Question 2: Dataset Analysis

1. Compute the average lesion coverage (percentage of pixels that are lesion) across all training images
2. What is the range of image resolutions in the dataset?
3. Show the histogram of lesion coverage. Are the masks balanced?

In [ ]:
# Analyze the training masks
coverages = []
resolutions = []
for img_path, mask_path in train_pairs:
    img = Image.open(img_path)
    mask = np.array(Image.open(mask_path).convert("L"))
    resolutions.append(img.size)  # (width, height)
    coverages.append((mask > 128).sum() / mask.size * 100)

print(f"Average lesion coverage: {np.mean(coverages):.1f}%")
print(f"Min/Max coverage: {np.min(coverages):.1f}% / {np.max(coverages):.1f}%")

# Resolution statistics
widths = [r[0] for r in resolutions]
heights = [r[1] for r in resolutions]
print(f"\nResolution range: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")

# Histogram
plt.figure(figsize=(8, 4))
plt.hist(coverages, bins=30, color="#2196F3", edgecolor="white")
plt.xlabel("Lesion Coverage (%)")
plt.ylabel("Number of Images")
plt.title("Distribution of Lesion Coverage in Training Set")
plt.tight_layout()
plt.show()

## 4. Preparing the Data

We'll use the Hugging Face `datasets` library to create a proper dataset, and `SegformerImageProcessor` to handle image preprocessing (resizing, normalization).

We'll resize all images to **256x256** pixels for training speed on Colab's free GPU.

In [ ]:
from datasets import Dataset, Image as HFImage
from transformers import SegformerImageProcessor

# Use a subset of training data for speed (adjust n_train as needed)
n_train = 600
selected_train = train_pairs[:n_train]

print(f"Using {len(selected_train)} training images and {len(val_pairs)} validation images")

# Create Hugging Face Datasets
train_ds = Dataset.from_dict({
    "image": [p[0] for p in selected_train],
    "label": [p[1] for p in selected_train],
}).cast_column("image", HFImage()).cast_column("label", HFImage())

val_ds = Dataset.from_dict({
    "image": [p[0] for p in val_pairs],
    "label": [p[1] for p in val_pairs],
}).cast_column("image", HFImage()).cast_column("label", HFImage())

print(f"Train dataset: {train_ds}")
print(f"Val dataset: {val_ds}")

In [ ]:
# Setup the image processor
processor = SegformerImageProcessor(
    do_resize=True,
    size={"height": 256, "width": 256},
    do_normalize=True,
    do_reduce_labels=False,
)

def preprocess(example):
    """Preprocess a single example: resize image and mask, normalize."""
    image = example["image"].convert("RGB")
    # Convert mask: 255 (white=lesion) -> 1, 0 (black=background) -> 0
    mask = np.array(example["label"].convert("L"), dtype=np.int64)
    mask = (mask > 128).astype(np.int64)  # Binarize
    inputs = processor(images=image, segmentation_maps=[mask], return_tensors="pt")
    return {k: v.squeeze(0) for k, v in inputs.items()}

# Apply transforms (lazy - applied on-the-fly when accessing items)
train_ds.set_transform(preprocess)
val_ds.set_transform(preprocess)

# Verify the output format
sample = train_ds[0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")  # [3, 256, 256]
print(f"labels shape: {sample['labels'].shape}")               # [256, 256]
print(f"Labels unique values: {sample['labels'].unique().tolist()}")

## 5. Fine-tuning SegFormer

[SegFormer](https://arxiv.org/abs/2105.15203) is an efficient transformer-based model for semantic segmentation. It combines a hierarchical **Mix Transformer (MiT)** encoder with a lightweight **MLP decoder**.

We load the `nvidia/mit-b0` backbone (pre-trained on ImageNet) and add a segmentation head with 2 output classes (background + lesion).

The Hugging Face `Trainer` handles the training loop, evaluation, checkpointing, and logging automatically.

In [ ]:
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=2,
    id2label={0: "background", 1: "lesion"},
    label2id={"background": 0, "lesion": 1},
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
import evaluate
from torch import nn

metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        # SegFormer outputs at 1/4 resolution, upsample to match labels
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)

        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=2,
            ignore_index=255,
            reduce_labels=False,
        )

        per_cat_iou = metrics.pop("per_category_iou").tolist()
        metrics.pop("per_category_accuracy")

        return {
            "mean_iou": metrics["mean_iou"],
            "lesion_iou": per_cat_iou[1],
        }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="segformer-skin-lesion",
    learning_rate=6e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="lesion_iou",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_accumulation_steps=5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Plot training history
log_history = trainer.state.log_history
train_losses = [(l["step"], l["loss"]) for l in log_history if "loss" in l]
eval_metrics = [(l["step"], l.get("eval_lesion_iou", 0)) for l in log_history if "eval_lesion_iou" in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

steps, losses = zip(*train_losses)
ax1.plot(steps, losses, color="#2196F3")
ax1.set_xlabel("Step")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")

if eval_metrics:
    e_steps, e_ious = zip(*eval_metrics)
    ax2.plot(e_steps, e_ious, color="#4CAF50", marker="o")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Lesion IoU")
    ax2.set_title(f"Validation Lesion IoU (best: {max(e_ious):.3f})")

plt.tight_layout()
plt.show()

## Question 3: Improve the Results

Try to improve the validation lesion IoU by experimenting with:

1. **More training data**: increase `n_train` (up to 2000)
2. **More epochs**: increase `num_train_epochs` to 20 or 30
3. **Learning rate**: try `3e-5` or `1e-4`
4. **Larger model**: replace `nvidia/mit-b0` with `nvidia/mit-b2` (more parameters)

Which change has the most impact?

In [ ]:
# Your experiments here
# Hint: you can re-run the training cells above with modified parameters
# or create a new Trainer with different settings

# Example: try a different learning rate
# training_args.learning_rate = ...
# trainer = Trainer(model=model, args=training_args, ...)
# trainer.train()
...

## 6. Evaluation and Visualization

Let's visualize the model's predictions on the validation set and compare them with the ground truth.

In [ ]:
def predict_segmentation(model, processor, image, device="cpu"):
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    pred = nn.functional.interpolate(
        logits.cpu(),
        size=image.size[::-1],  # (height, width)
        mode="bilinear",
        align_corners=False,
    ).argmax(dim=1).squeeze().numpy()
    return pred


def visualize_predictions(model, processor, pairs, indices, device="cpu"):
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = [axes]

    for row, idx in enumerate(indices):
        img_path, mask_path = pairs[idx]
        image = Image.open(img_path).convert("RGB")
        gt_mask = np.array(Image.open(mask_path).convert("L"))
        gt_binary = (gt_mask > 128).astype(np.uint8)

        pred = predict_segmentation(model, processor, image, device)

        iou = compute_iou(pred, gt_binary)
        dice = compute_dice(pred, gt_binary)

        # Image
        axes[row][0].imshow(image)
        axes[row][0].set_title("Image")

        # Ground truth
        axes[row][1].imshow(gt_binary, cmap="gray")
        axes[row][1].set_title("Ground Truth")

        # Prediction
        axes[row][2].imshow(pred, cmap="gray")
        axes[row][2].set_title(f"Prediction\nIoU: {iou:.2f} | Dice: {dice:.2f}")

        # Overlay: green = correct, red = false positive, blue = false negative
        overlay = np.zeros((*gt_binary.shape, 3))
        overlay[np.logical_and(pred == 1, gt_binary == 1)] = [0, 1, 0]  # True positive
        overlay[np.logical_and(pred == 1, gt_binary == 0)] = [1, 0, 0]  # False positive
        overlay[np.logical_and(pred == 0, gt_binary == 1)] = [0, 0, 1]  # False negative
        axes[row][3].imshow(overlay)
        axes[row][3].set_title("Error Map (G=TP, R=FP, B=FN)")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

# Move model to CPU for inference
model_device = "cpu"
model.to(model_device)
model.eval()

visualize_predictions(model, processor, val_pairs, [0, 5, 10, 15, 20], model_device)

In [ ]:
# Compute IoU for all validation images
print("Evaluating on full validation set...\n")
all_ious = []
all_dices = []

for img_path, mask_path in val_pairs:
    image = Image.open(img_path).convert("RGB")
    gt_mask = (np.array(Image.open(mask_path).convert("L")) > 128).astype(np.uint8)
    pred = predict_segmentation(model, processor, image, model_device)
    all_ious.append(compute_iou(pred, gt_mask))
    all_dices.append(compute_dice(pred, gt_mask))

print(f"Validation results ({len(val_pairs)} images):")
print(f"  Mean IoU:  {np.mean(all_ious):.3f} +/- {np.std(all_ious):.3f}")
print(f"  Mean Dice: {np.mean(all_dices):.3f} +/- {np.std(all_dices):.3f}")

# Distribution of IoU scores
plt.figure(figsize=(8, 4))
plt.hist(all_ious, bins=25, color="#4CAF50", edgecolor="white")
plt.axvline(np.mean(all_ious), color="red", linestyle="--", label=f"Mean: {np.mean(all_ious):.3f}")
plt.xlabel("IoU Score")
plt.ylabel("Number of Images")
plt.title("Distribution of IoU Scores on Validation Set")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show best and worst predictions
sorted_indices = np.argsort(all_ious)

print("Best predictions:")
visualize_predictions(model, processor, val_pairs, sorted_indices[-3:][::-1], model_device)

print("Worst predictions:")
visualize_predictions(model, processor, val_pairs, sorted_indices[:3], model_device)

## Question 4: Error Analysis

Look at the worst predictions above.

1. What patterns do you see in the images where the model fails?
2. Do the failures have anything in common (image quality, lesion size, color)?
3. How could you improve the model's performance on these difficult cases? Think about:
   - Data augmentation (what augmentations would help?)
   - Using a larger model backbone
   - Using more training data
   - Pre-processing the images

In [ ]:
# Analyze error patterns
# Which images have IoU < 0.5?
poor_predictions = [(i, iou) for i, iou in enumerate(all_ious) if iou < 0.5]
print(f"Images with IoU < 0.5: {len(poor_predictions)} / {len(all_ious)}")

# Show the distribution of lesion coverage for poor vs good predictions
poor_coverages = []
good_coverages = []
for i, iou in enumerate(all_ious):
    mask = np.array(Image.open(val_pairs[i][1]).convert("L"))
    coverage = (mask > 128).sum() / mask.size
    if iou < 0.5:
        poor_coverages.append(coverage)
    else:
        good_coverages.append(coverage)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(good_coverages, bins=20, alpha=0.7, label=f"IoU >= 0.5 (n={len(good_coverages)})", color="#4CAF50")
ax.hist(poor_coverages, bins=20, alpha=0.7, label=f"IoU < 0.5 (n={len(poor_coverages)})", color="#F44336")
ax.set_xlabel("Lesion Coverage")
ax.set_ylabel("Count")
ax.set_title("Lesion Coverage: Good vs Poor Predictions")
ax.legend()
plt.tight_layout()
plt.show()

## 7. What's Next

In this notebook you learned to:
- Use the **Hugging Face ecosystem** (Hub, `pipeline()`, `transformers`, `datasets`, `evaluate`)
- Understand **image segmentation** concepts (semantic, instance, IoU, Dice)
- **Fine-tune SegFormer** on a medical imaging dataset (ISIC skin lesions)

To go further:

- **Bigger models**: try `nvidia/mit-b2` or `nvidia/mit-b5` for better accuracy
- **Data augmentation**: add random flips, rotations, color jitter using `torchvision.transforms`
- **Other segmentation models**: [Mask2Former](https://huggingface.co/docs/transformers/model_doc/mask2former), [SAM (Segment Anything)](https://huggingface.co/docs/transformers/model_doc/sam), [OneFormer](https://huggingface.co/docs/transformers/model_doc/oneformer)
- **Other medical datasets**: [DRIVE](https://drive.grand-challenge.org/) (retinal vessels), [BRATS](https://www.med.upenn.edu/cbica/brats2021/) (brain tumors)
- **Related notebooks**: [Open Vocabulary Detection](./Open_Vocabulary_Detection.ipynb) (Grounded-SAM for text-prompted segmentation)
- [HuggingFace Segmentation Guide](https://huggingface.co/docs/transformers/tasks/semantic_segmentation)